In [5]:
import os
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import KFold, cross_validate


PROJET = Path("/info/etu/m1/s2603634/Projet/ChallengeMachineLearning")
if str(PROJET) not in sys.path:
    sys.path.insert(0, str(PROJET))

from automl.data import Dataset, lireDataset, listerDatasets

DATA_ROOT = Path("/info/corpus/ChallengeMachineLearning")

In [ ]:
TYPES_VALIDES = {"numerical": "Numerical", "categorical": "Categorical", "binary": "Binary"}


def _inf_vers_nan(X):
    """Remplace +/-inf par NaN sans toucher aux zéros implicites d'une matrice sparse."""
    if sparse.issparse(X):
        X = X.tocsr(copy=True)
        X.data = X.data.astype(float, copy=False)
        X.data[np.isinf(X.data)] = np.nan
        return X
    X = np.array(X, dtype=float, copy=True)
    X[np.isinf(X)] = np.nan
    return X


class PretraitementAuto(BaseEstimator, TransformerMixin):
    """Prétraitement piloté par les types du fichier .type (Numerical / Categorical / Binary)"""

    def __init__(self, types, strategie_cat="frequente", mise_a_echelle=False, sortie_creuse=None):
        self.types = types
        self.strategie_cat = strategie_cat
        self.mise_a_echelle = mise_a_echelle
        self.sortie_creuse = sortie_creuse

    def _indices_par_type(self, n_colonnes):
        if len(self.types) != n_colonnes:
            raise ValueError(
                f"{len(self.types)} types déclarés pour {n_colonnes} colonnes : incohérence."
            )
        normalises = []
        for t in self.types:
            cle = str(t).strip().casefold()
            if cle not in TYPES_VALIDES:
                raise ValueError(f"Type de colonne inconnu : {t!r} (attendu : {sorted(TYPES_VALIDES.values())})")
            normalises.append(TYPES_VALIDES[cle])
        return {
            nom: [i for i, t in enumerate(normalises) if t == nom]
            for nom in TYPES_VALIDES.values()
        }

    def fit(self, X, y=None):
        if self.strategie_cat not in ("frequente", "manquant"):
            raise ValueError("strategie_cat doit valoir 'frequente' ou 'manquant'.")
        if X.ndim != 2:
            raise ValueError("X doit être une matrice 2D.")

        X = _inf_vers_nan(X)
        self.n_features_in_ = X.shape[1]
        self.entree_creuse_ = sparse.issparse(X)
        self.sortie_creuse_ = self.entree_creuse_ if self.sortie_creuse is None else bool(self.sortie_creuse)
        idx = self._indices_par_type(self.n_features_in_)

        etapes_num = [("imputer", SimpleImputer(strategy="median", keep_empty_features=True))]
        if self.mise_a_echelle:
            etapes_num.append(("echelle", StandardScaler(with_mean=not self.entree_creuse_)))

        etapes_bin = [("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True))]

        if self.strategie_cat == "frequente":
            imp_cat = SimpleImputer(strategy="most_frequent", keep_empty_features=True)
        else:
            imp_cat = SimpleImputer(strategy="constant", fill_value=-1, keep_empty_features=True)
        etapes_cat = [
            ("imputer", imp_cat),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]

        blocs = []
        for nom, etapes, cols in (
            ("num", etapes_num, idx["Numerical"]),
            ("bin", etapes_bin, idx["Binary"]),
            ("cat", etapes_cat, idx["Categorical"]),
        ):
            if cols:
                blocs.append((nom, Pipeline(etapes), cols))

        if not blocs:
            raise ValueError("Aucune colonne à traiter.")

        self.transformeur_ = ColumnTransformer(
            blocs, remainder="drop", sparse_threshold=1.0 if self.sortie_creuse_ else 0.0
        )
        self.transformeur_.fit(X)
        return self

    def transform(self, X):
        check_is_fitted(self, "transformeur_")
        if X.shape[1] != self.n_features_in_:
            raise ValueError(
                f"X a {X.shape[1]} colonnes, le prétraitement en a appris {self.n_features_in_}."
            )
        Xt = self.transformeur_.transform(_inf_vers_nan(X))
        if self.sortie_creuse_ and not sparse.issparse(Xt):
            Xt = sparse.csr_matrix(Xt)
        elif not self.sortie_creuse_ and sparse.issparse(Xt):
            Xt = Xt.toarray()  # choix explicite de l'appelant : à n'utiliser que si la taille le permet
        return Xt

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "transformeur_")
        return self.transformeur_.get_feature_names_out()

In [3]:
def jeu_mixte(n, graine):
    """Données synthétiques : 3 numériques, 2 catégorielles (codes), 1 binaire."""
    rng = np.random.default_rng(graine)
    X = np.column_stack([
        rng.normal(50, 10, n), rng.exponential(5, n), rng.integers(0, 1000, n).astype(float),
        rng.integers(0, 4, n).astype(float), rng.integers(0, 6, n).astype(float),
        rng.integers(0, 2, n).astype(float),
    ])
    return X


TYPES = ["Numerical"] * 3 + ["Categorical"] * 2 + ["Binary"]

try:
    PretraitementAuto(types=TYPES[:-1]).fit(jeu_mixte(50, 0))
    raise AssertionError("une erreur était attendue")
except ValueError as e:
    print("1. OK, erreur claire :", e)

X = jeu_mixte(200, 1)
X[::7, 0] = np.nan; X[3, 1] = np.inf; X[5, 1] = -np.inf; X[::11, 3] = np.nan; X[::13, 5] = np.nan
pre = PretraitementAuto(types=TYPES).fit(X)
Xt = pre.transform(X)
assert np.isfinite(Xt.toarray() if sparse.issparse(Xt) else Xt).all()
print("2. OK, sortie finie, forme", Xt.shape)

X_train, X_test = jeu_mixte(300, 2), jeu_mixte(100, 3) + 1000
pre = PretraitementAuto(types=TYPES).fit(X_train)
mediane_apprise = pre.transformeur_.named_transformers_["num"]["imputer"].statistics_
assert np.allclose(mediane_apprise, np.median(X_train[:, :3], axis=0))
pre.transform(X_test)
assert np.allclose(pre.transformeur_.named_transformers_["num"]["imputer"].statistics_, mediane_apprise)
print("3. OK, médianes = celles du train, inchangées après transform(test)")

X_valid = jeu_mixte(37, 4); X_valid[:, 3] = 99
assert pre.transform(X_valid).shape[1] == pre.transform(X_train).shape[1]
print("4. OK, dimension stable avec catégorie inconnue :", pre.transform(X_valid).shape)

X5 = jeu_mixte(100, 5); X5[:, 0] = np.nan; X5[:, 3] = np.nan
Xt5 = PretraitementAuto(types=TYPES).fit(X5).transform(X5)
assert np.isfinite(Xt5.toarray() if sparse.issparse(Xt5) else Xt5).all()
print("5. OK, colonnes 100 % NaN gérées, forme", Xt5.shape)

X6 = jeu_mixte(120, 6); X6[::5, 5] = np.nan
pre6 = PretraitementAuto(types=TYPES).fit(X6)
bloc_bin = pre6.transformeur_.named_transformers_["bin"].transform(X6[:, [5]])
assert set(np.unique(bloc_bin)) <= {0.0, 1.0}
print("6. OK, binaire reste dans {0, 1}")

a = PretraitementAuto(types=TYPES, strategie_cat="manquant").fit(X, y=None).transform(X)
b = PretraitementAuto(types=TYPES, strategie_cat="manquant").fit(X, y=None).transform(X)
assert (a != b).nnz == 0 if sparse.issparse(a) else np.array_equal(a, b)
print("7. OK, sans labels et reproductible")

rng = np.random.default_rng(8)
n, d = 2000, 50_000
Xs = sparse.random(n, d, density=0.001, format="csr", random_state=8)
Xs.data[:5] = np.inf
pre_s = PretraitementAuto(types=["Numerical"] * d, mise_a_echelle=True)
t0 = time.time()
Xst = pre_s.fit(Xs).transform(Xs)
assert sparse.issparse(Xst) and Xst.shape == (n, d) and np.isfinite(Xst.data).all()
print(f"8. OK, sparse conservé, forme {Xst.shape}, nnz={Xst.nnz}, {time.time()-t0:.1f}s")

y = (jeu_mixte(400, 9)[:, 5] + np.random.default_rng(9).integers(0, 2, 400) > 0).astype(int)
Xcv = jeu_mixte(400, 9); Xcv[::9, 3] = np.nan
modele = make_pipeline(PretraitementAuto(types=TYPES, mise_a_echelle=True), LogisticRegression(max_iter=500))
clone(modele)
scores = cross_val_score(modele, Xcv, y, cv=4)
print("9. OK, Pipeline + CV exécutés ; scores de plis :", np.round(scores, 3), "(données synthétiques, sans valeur de performance)")

1. OK, erreur claire : 5 types déclarés pour 6 colonnes : incohérence.
2. OK, sortie finie, forme (200, 14)
3. OK, médianes = celles du train, inchangées après transform(test)
4. OK, dimension stable avec catégorie inconnue : (37, 14)
5. OK, colonnes 100 % NaN gérées, forme (100, 11)
6. OK, binaire reste dans {0, 1}
7. OK, sans labels et reproductible
8. OK, sparse conservé, forme (2000, 50000), nnz=100000, 1.9s
9. OK, Pipeline + CV exécutés ; scores de plis : [0.75 0.79 0.77 0.78] (données synthétiques, sans valeur de performance)


In [4]:
N_JOBS = int(os.environ.get("AUTOML_N_JOBS", 4))  # jamais -1
GRAINE = 42
N_TRAIN_MAX = 8000  # sous-échantillon du train pour aller vite (valid et test restent intacts)

data_A = lireDataset(DATA_ROOT / "data_A")
data_A.split()
rng = np.random.default_rng(GRAINE)
n = data_A.train.x.shape[0]
idx = np.sort(rng.choice(n, size=min(N_TRAIN_MAX, n), replace=False))
x_train, y_train = data_A.train.x[idx], data_A.train.y[idx].astype(int)
x_valid, y_valid = data_A.valid.x, data_A.valid.y.astype(int)

print("train utilisé :", x_train.shape, "| valid :", x_valid.shape, "| étiquettes :", y_train.shape[1])
print("fréquence de chaque étiquette (train) :", y_train.mean(axis=0).round(3))

types_A = data_A.types
variantes = {
    "A. tout numérique (référence naïve)": dict(types=["Numerical"] * len(types_A)),
    "B. catégories, NaN -> modalité fréquente": dict(types=types_A, strategie_cat="frequente"),
    "C. catégories, NaN -> modalité 'manquant'": dict(types=types_A, strategie_cat="manquant"),
}
modeles = {
    "ForêtAléatoire": (lambda: RandomForestClassifier(
        n_estimators=50, min_samples_leaf=2, random_state=GRAINE, n_jobs=N_JOBS), False),
    "RégLogistique (1 contre tous)": (lambda: OneVsRestClassifier(
        LogisticRegression(max_iter=300), n_jobs=N_JOBS), True),
}

lignes = []
for nom_mod, (fabrique, echelle) in modeles.items():  # d'abord la forêt (rapide), puis la logistique
    for nom_var, params in variantes.items():
        pipe = make_pipeline(PretraitementAuto(mise_a_echelle=echelle, **params), fabrique())
        t0 = time.time()
        pipe.fit(x_train, y_train)
        duree = time.time() - t0
        pred = pipe.predict(x_valid)
        lignes.append({
            "variante": nom_var, "modèle": nom_mod,
            "colonnes_après_prétraitement": len(pipe[0].get_feature_names_out()),
            "F1 pondéré": round(f1_score(y_valid, pred, average="weighted", zero_division=0), 4),
            "F1 micro": round(f1_score(y_valid, pred, average="micro", zero_division=0), 4),
            "durée_fit (s)": round(duree, 1),
        })
        print(f"terminé : {nom_mod} | {nom_var} | {duree:.1f}s", flush=True)

resultats_A = pd.DataFrame(lignes).sort_values(["modèle", "F1 pondéré"], ascending=[True, False])
display(resultats_A)

train utilisé : (8000, 24) | valid : (3419, 24) | étiquettes : 3
fréquence de chaque étiquette (train) : [0.852 0.666 0.764]
terminé : ForêtAléatoire | A. tout numérique (référence naïve) | 0.4s
terminé : ForêtAléatoire | B. catégories, NaN -> modalité fréquente | 0.4s
terminé : ForêtAléatoire | C. catégories, NaN -> modalité 'manquant' | 0.4s
terminé : RégLogistique (1 contre tous) | A. tout numérique (référence naïve) | 2.4s
terminé : RégLogistique (1 contre tous) | B. catégories, NaN -> modalité fréquente | 2.0s
terminé : RégLogistique (1 contre tous) | C. catégories, NaN -> modalité 'manquant' | 2.8s


,variante,modèle,colonnes_après_prétraitement,F1 pondéré,F1 micro,durée_fit (s)
2,"C. catégories, NaN -> modalité 'manquant'",ForêtAléatoire,218,0.9056,0.9063,0.4
1,"B. catégories, NaN -> modalité fréquente",ForêtAléatoire,212,0.9044,0.9050,0.4
0,A. tout numérique (référence naïve),ForêtAléatoire,24,0.9008,0.9017,0.4
5,"C. catégories, NaN -> modalité 'manquant'",RégLogistique (1 contre tous),218,0.9054,0.9062,2.8
4,"B. catégories, NaN -> modalité fréquente",RégLogistique (1 contre tous),212,0.9043,0.9051,2.0
3,A. tout numérique (référence naïve),RégLogistique (1 contre tous),24,0.8788,0.8788,2.4


In [6]:
N_CV_MAX = 12000  # mettre None pour utiliser tout le train

x_cv, y_cv = data_A.train.x, data_A.train.y.astype(int)
if N_CV_MAX is not None and x_cv.shape[0] > N_CV_MAX:
    idx_cv = np.sort(np.random.default_rng(GRAINE).choice(x_cv.shape[0], size=N_CV_MAX, replace=False))
    x_cv, y_cv = x_cv[idx_cv], y_cv[idx_cv]
print("données de la CV :", x_cv.shape, "| le test final n'est pas utilisé")

decoupage = KFold(n_splits=5, shuffle=True, random_state=GRAINE)  # mêmes plis pour tous les modèles
metriques = ["f1_weighted", "f1_micro"]

lignes_cv = []

base = DummyClassifier(strategy="constant", constant=[1] * y_cv.shape[1])
r = cross_validate(base, x_cv, y_cv, cv=decoupage, scoring=metriques)
lignes_cv.append({"variante": "(baseline : toujours 1)", "modèle": "-",
                  "F1 pondéré moy.": r["test_f1_weighted"].mean(), "écart-type": r["test_f1_weighted"].std(),
                  "F1 micro moy.": r["test_f1_micro"].mean(), "durée (s)": 0.0})

for nom_mod, (fabrique, echelle) in modeles.items():
    for nom_var, params in variantes.items():
        pipe = make_pipeline(PretraitementAuto(mise_a_echelle=echelle, **params), fabrique())
        t0 = time.time()
        r = cross_validate(pipe, x_cv, y_cv, cv=decoupage, scoring=metriques)  # prétraitement réappris dans CHAQUE pli
        duree = time.time() - t0
        lignes_cv.append({"variante": nom_var, "modèle": nom_mod,
                          "F1 pondéré moy.": r["test_f1_weighted"].mean(), "écart-type": r["test_f1_weighted"].std(),
                          "F1 micro moy.": r["test_f1_micro"].mean(), "durée (s)": duree})
        print(f"terminé : {nom_mod} | {nom_var} | {duree:.1f}s", flush=True)

resultats_cv = pd.DataFrame(lignes_cv).round(4)
display(resultats_cv)

données de la CV : (12000, 24) | le test final n'est pas utilisé
terminé : ForêtAléatoire | A. tout numérique (référence naïve) | 2.3s
terminé : ForêtAléatoire | B. catégories, NaN -> modalité fréquente | 3.1s
terminé : ForêtAléatoire | C. catégories, NaN -> modalité 'manquant' | 3.0s
terminé : RégLogistique (1 contre tous) | A. tout numérique (référence naïve) | 5.7s
terminé : RégLogistique (1 contre tous) | B. catégories, NaN -> modalité fréquente | 17.2s
terminé : RégLogistique (1 contre tous) | C. catégories, NaN -> modalité 'manquant' | 11.6s


,variante,modèle,F1 pondéré moy.,écart-type,F1 micro moy.,durée (s)
0,(baseline : toujours 1),-,0.8671,0.0021,0.8642,0.0000
1,A. tout numérique (référence naïve),ForêtAléatoire,0.9026,0.0013,0.9033,2.2741
2,"B. catégories, NaN -> modalité fréquente",ForêtAléatoire,0.9035,0.0012,0.9041,3.0601
3,"C. catégories, NaN -> modalité 'manquant'",ForêtAléatoire,0.9054,0.0018,0.9060,3.0396
4,A. tout numérique (référence naïve),RégLogistique (1 contre tous),0.8794,0.0020,0.8790,5.6863
5,"B. catégories, NaN -> modalité fréquente",RégLogistique (1 contre tous),0.9044,0.0021,0.9051,17.1798
6,"C. catégories, NaN -> modalité 'manquant'",RégLogistique (1 contre tous),0.9056,0.0022,0.9064,11.6227
